# Credit Card Fraud Detection Using Machine Learning

## Objective

The goal of this project is to detect **fraudulent credit card transactions** using machine learning.

Most transactions are genuine, but a few are fraud. We will train a model to automatically tell them apart.

**Dataset:** Kaggle Credit Card Fraud Detection Dataset
- Columns: `Time`, `V1` to `V28` (anonymized), `Amount`, and `Class`
- `Class = 0` → Normal transaction
- `Class = 1` → Fraudulent transaction


## Step 1: Download the Dataset

We use the `kagglehub` library to download the dataset directly from Kaggle.

> **First time setup:** You will need a free Kaggle account. When prompted, enter your Kaggle username and API key.
> You can find your API key at: https://www.kaggle.com/settings → "Create New Token"


In [ ]:
# Install kagglehub (only needed once)
!pip install kagglehub -q


In [ ]:
# Download the dataset from Kaggle
import kagglehub
import shutil
import os

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print("Dataset downloaded to:", path)

# Copy creditcard.csv to the current folder (if not already there)
src = os.path.join(path, "creditcard.csv")
dst = "creditcard.csv"
if not os.path.exists(dst):
    shutil.copy(src, dst)
    print("Copied creditcard.csv to current folder.")
else:
    print("creditcard.csv already exists in current folder.")


## Step 2: Import Libraries


In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, classification_report)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")


## Step 3: Load the Dataset


In [ ]:
# Load the CSV file
df = pd.read_csv('creditcard.csv')
print("Dataset loaded. Total rows:", len(df))


## Step 4: Basic Data Analysis

Let's understand the dataset before building any model.


In [ ]:
# Check the shape (rows, columns)
print("Shape:", df.shape)


In [ ]:
# Display first 5 rows
df.head()


In [ ]:
# Check for missing values
print("Missing values:", df.isnull().sum().sum())


In [ ]:
# Check data types
df.dtypes


In [ ]:
# Count fraud vs normal transactions
print(df['Class'].value_counts())
print()
print("Fraud percentage:", round(df['Class'].mean() * 100, 2), "%")


## Step 5: Exploratory Data Analysis (EDA)

Let's visualize the data using simple graphs.


In [ ]:
# Plot 1: Class distribution (Fraud vs Normal)
sns.countplot(x='Class', data=df)
plt.title('Class Distribution')
plt.xticks([0, 1], ['Normal', 'Fraud'])
plt.show()


In [ ]:
# Plot 2: Transaction amount distribution
plt.hist(df['Amount'], bins=50, edgecolor='black')
plt.title('Transaction Amount Distribution')
plt.xlabel('Amount')
plt.ylabel('Count')
plt.xlim(0, 500)  # Zoom in for better view
plt.show()


In [ ]:
# Plot 3: Compare fraud vs normal amounts using box plot
sns.boxplot(x='Class', y='Amount', data=df)
plt.title('Transaction Amount: Normal vs Fraud')
plt.xticks([0, 1], ['Normal', 'Fraud'])
plt.ylim(0, 500)
plt.show()


## Step 6: Understanding Class Imbalance

Our dataset has about **99.8% normal** and only **0.2% fraud** transactions. This is called **class imbalance**.

**Why is this a problem?**
- If a model always predicts "Normal", it will be 99.8% accurate — but it catches zero frauds!
- So accuracy alone can be misleading.
- We need other metrics like **Precision**, **Recall**, and **F1-Score** to properly evaluate the model.


## Step 7: Data Preprocessing

We need to prepare the data before training:
1. Scale the `Amount` column (so all features are on a similar range)
2. Drop the `Time` column (not useful)
3. Split into training (80%) and testing (20%) sets


In [ ]:
# Scale the Amount column
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])

# Drop the Time column
df = df.drop('Time', axis=1)

# Separate features (X) and target (y)
X = df.drop('Class', axis=1)
y = df['Class']

print("Features shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape[0], "rows")
print("Testing set:", X_test.shape[0], "rows")


## Step 8: Train Machine Learning Models

We will train two simple models:
1. **Logistic Regression** — A simple model that predicts probabilities
2. **Decision Tree** — A model that makes decisions like a flowchart


In [ ]:
# Model 1: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
print("Logistic Regression trained.")


In [ ]:
# Model 2: Decision Tree
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
print("Decision Tree trained.")


## Step 9: Model Evaluation

We evaluate using:
- **Accuracy** — How many predictions were correct overall
- **Precision** — When model says fraud, how often is it correct
- **Recall** — Out of all real frauds, how many did the model catch
- **F1-Score** — A balance between precision and recall
- **Confusion Matrix** — Shows correct/incorrect predictions in a table


In [ ]:
# Evaluate Logistic Regression
print("=== Logistic Regression ===")
print("Accuracy :", round(accuracy_score(y_test, lr_pred), 4))
print("Precision:", round(precision_score(y_test, lr_pred), 4))
print("Recall   :", round(recall_score(y_test, lr_pred), 4))
print("F1-Score :", round(f1_score(y_test, lr_pred), 4))
print()
print(classification_report(y_test, lr_pred, target_names=['Normal', 'Fraud']))


In [ ]:
# Evaluate Decision Tree
print("=== Decision Tree ===")
print("Accuracy :", round(accuracy_score(y_test, dt_pred), 4))
print("Precision:", round(precision_score(y_test, dt_pred), 4))
print("Recall   :", round(recall_score(y_test, dt_pred), 4))
print("F1-Score :", round(f1_score(y_test, dt_pred), 4))
print()
print(classification_report(y_test, dt_pred, target_names=['Normal', 'Fraud']))


In [ ]:
# Confusion Matrix - Logistic Regression
sns.heatmap(confusion_matrix(y_test, lr_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
# Confusion Matrix - Decision Tree
sns.heatmap(confusion_matrix(y_test, dt_pred), annot=True, fmt='d', cmap='Oranges')
plt.title('Confusion Matrix - Decision Tree')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


## Step 10: Why Accuracy Alone Is Not Enough

In fraud detection, the dataset is highly imbalanced (99.8% normal, 0.2% fraud).

If a model predicts **everything as normal**, it still gets **99.8% accuracy** — but catches **zero frauds**.

That's why we use:
- **Recall** — Did we catch all the frauds? (Most important for fraud detection)
- **Precision** — Are our fraud alerts actually correct?
- **F1-Score** — A single number balancing both

**Key takeaway:** For fraud detection, **recall matters more than accuracy**.


## Step 11: Conclusion

In this project, we:
1. Loaded and explored a credit card fraud dataset
2. Visualized the data and understood class imbalance
3. Preprocessed the data (scaling and splitting)
4. Trained two models: Logistic Regression and Decision Tree
5. Evaluated both models using multiple metrics

**Key Learning:** Accuracy alone is not enough for imbalanced datasets. Always check Precision, Recall, and F1-Score.

**Future Improvements:**
- Try Random Forest or XGBoost
- Use SMOTE to handle class imbalance
- Tune model hyperparameters
